# Experiment Plotting v2

Gathers computed results and produces plots/tables for the paper, reading from the
confidence-threshold-tagged experiment pipeline (`experiments/<dataset>/{alphas,max_rules,lambda}.py`,
see `experiments/README.md` and `experiments/aniso/run_confidence_sweep.sh`).

**Data availability note:** as of writing, only `aniso` has been run through the confidence-tagged
pipeline end to end. Every loader below is defensive (skips a missing dataset/threshold with a
printed notice rather than raising), so every section will render with as few as one dataset column
until the other datasets are (re)run with `--confidence`. No code changes are needed when that
happens -- just add the dataset name to `DATASETS` / `LAMBDA_DATASETS` below if it isn't already
listed.

## Setup

### Imports & Style

In [ ]:
import sys
sys.path.insert(0, '..')

import json
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from matplotlib.collections import LineCollection
from matplotlib.ticker import MaxNLocator, FormatStrFormatter
import matplotlib.lines as mlines
import matplotlib.cm as cm
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
import seaborn as sns
from sklearn.cluster import KMeans, DBSCAN
from data.preprocessing import *
from intercluster.plotting import *
from intercluster import *
from intercluster.decision_trees import *
from intercluster.decision_sets import *
from intercluster.decision_sets.mining import *
from intercluster.decision_sets.objectives import *

%load_ext autoreload
%autoreload 2

In [ ]:
# This assumes tex is installed in your system,
# if not you may simply remove most of this aside from font.size
# To get tex working on linux run the following:
# `sudo apt-get install texlive-latex-extra texlive-fonts-recommended dvipng cm-super`
plt.rcParams.update({
    "pgf.texsystem": "pdflatex",
    "font.family": "serif",
    "font.serif": [],
    "text.usetex": True,
    "pgf.rcfonts": False,
    "font.size": 32
})

palette = sns.color_palette("husl", 8)
cmap = ListedColormap(palette)

### Color / Title / Marker Dictionaries

In [ ]:
# Comparison-model set: Decision-Tree, ExKMC, WRA, IDS, CBA, CN2, vs. PEC
# (unweighted and weighted). Colors are assigned with a step-3 stride over the
# 8 husl hues (gcd(3,8)=1, so consecutive assignments are 135 degrees apart in
# hue rather than 45), and every model additionally gets a distinct marker
# shape / hatch pattern as a second visual channel.
color_dict = {
    'Decision-Tree': cmap(5),
    'ExKMC': cmap(1),
    #'WRA': cmap(6),
    'IDS': cmap(0),
    'CBA': cmap(3),
    'CN2': cmap(7),
    'dscluster; ensemble': cmap(6),
    #'dscluster; ensemble; weighted': cmap(2),
    'Reference': 'black',
}

# Lazy-greedy / distorted-greedy (see the Lambdas section) share PEC's color --
# they're the same underlying model -- and are distinguished by linestyle
# instead, since distorted-greedy is only valid for lambda >= lambda_star.
linestyle_dict = {
    'dscluster; ensemble': 'solid',
    'dscluster; ensemble; weighted': 'solid',
    'dscluster; ensemble; lazy-greedy': 'solid',
    'dscluster; ensemble; distorted-greedy': 'dotted',
    'Decision-Tree': 'solid',
    'ExKMC': 'solid',
    'WRA': 'solid',
    'CBA': 'solid',
    'CN2': 'solid',
    'IDS': 'solid',
    'Reference': 'dashed',
}

hatch_dict = {
    'Decision-Tree': '',
    'ExKMC': '',
    'WRA': '',
    'CBA': '',
    'CN2': '',
    'IDS': '',
    'dscluster; ensemble': '//',
    'dscluster; ensemble; weighted': '',
}

marker_style_dict = {
    'dscluster; ensemble': 'D',
    #'dscluster; ensemble; weighted': 'd',
    'Decision-Tree': 'o',
    'ExKMC': 'o',
    'WRA': 'o',
    'CBA': 'o',
    'CN2': 'o',
    'IDS': 'o',
}

title_dict = {
    'dscluster; ensemble': r'\texttt{PEC}',
    'dscluster; ensemble; weighted': r'\texttt{PEC Weighted}',
    'Decision-Tree': r'\texttt{Decision-Tree}',
    'ExKMC': r'\texttt{ExKMC}',
    'WRA': r'\texttt{WRA}',
    'CBA': r'\texttt{CBA}',
    'CN2': r'\texttt{CN2}',
    'IDS': r'\texttt{IDS}',
}

objective_name_dict = {
    'coverage-cost': r'\textit{k-Means}',
    'coverage-mistake': r'\textit{Mistakes}',
    'coverage-pairwise-distance': r'\textit{Pairwise Distance}',
}

# Canonical comparison-model set and plot order, used by every section below
# instead of each section re-declaring its own near-duplicate local list (as
# the original notebook did in the Max Rules / Bicriteria / Confidence
# sections independently).
comparison_modules = ['Decision-Tree', 'ExKMC', 'IDS', 'CBA', 'CN2']
module_order = comparison_modules + ['dscluster; ensemble']

# {objective row index -> single-letter subscript used in y-axis math labels}
function_name_dict = {0: 'K', 1: 'M', 2: 'P'}

### Objective Reward/Cost Mapping

In [ ]:
objective_cost_reward_dict = {
    'coverage-mistake': {'reward': 'cluster-coverage', 'cost': 'mistakes'},
    'total-coverage-mistake': {'reward': 'total-coverage', 'cost': 'mistakes'},
    'coverage-cost': {'reward': 'cluster-coverage', 'cost': 'rule-clustering-cost'},
    'total-coverage-cost': {'reward': 'total-coverage', 'cost': 'rule-clustering-cost'},
    'coverage-pairwise-distance': {'reward': 'cluster-coverage', 'cost': 'rule-pairwise-distance'},
    'total-coverage-pairwise-distance': {'reward': 'total-coverage', 'cost': 'rule-pairwise-distance'},
    'coverage-mistake-weighted': {'reward': 'cluster-coverage', 'cost': 'mistakes'},
    'total-coverage-mistake-weighted': {'reward': 'total-coverage', 'cost': 'mistakes'},
    'coverage-cost-weighted': {'reward': 'cluster-coverage', 'cost': 'rule-clustering-cost'},
    'total-coverage-cost-weighted': {'reward': 'total-coverage', 'cost': 'rule-clustering-cost'},
    'coverage-pairwise-distance-weighted': {'reward': 'cluster-coverage', 'cost': 'rule-pairwise-distance'},
    'total-coverage-pairwise-distance-weighted': {'reward': 'total-coverage', 'cost': 'rule-pairwise-distance'},
}

### Axis-Limit Helpers

In [ ]:
# Helpers for picking "nice" axis limits and ticks for our plots. We want the
# limits and ticks to be "nice" numbers (e.g. multiples of 1, 2, or 5) and to
# have exactly 3 ticks (the endpoints and the midpoint), so plots look clean
# and interpretable.
def _nice_step(x: float) -> float:
    """Return a 'nice' step size near x using the 1-2-5 rule."""
    if not np.isfinite(x) or x <= 0:
        return 1.0
    exp = np.floor(np.log10(x))
    f = x / (10 ** exp)
    if f <= 1:
        nf = 1
    elif f <= 2:
        nf = 2
    elif f <= 5:
        nf = 5
    else:
        nf = 10
    return nf * (10 ** exp)


def nice_lim_for_3_ticks(raw_min: float, raw_max: float, *, clip=(0.0, 1.0)):
    """Choose ymin/ymax so that 3 evenly spaced ticks are also 'nice'.

    We enforce: ticks = [ymin, ymin+step, ymin+2*step], so both ends and the midpoint
    are multiples of a 'nice' step.
    """
    lo_clip, hi_clip = clip

    raw_min = max(raw_min, lo_clip)
    raw_max = min(raw_max, hi_clip)

    if not (np.isfinite(raw_min) and np.isfinite(raw_max)):
        return lo_clip, hi_clip, np.array([lo_clip, (lo_clip + hi_clip) / 2, hi_clip])

    if raw_max <= raw_min:
        # Degenerate case: build a small nice range around raw_min
        step = _nice_step(abs(raw_min) * 0.1 + 1e-3)
        ymin = np.floor(raw_min / step) * step
        ymax = ymin + 2 * step
    else:
        # With 3 ticks, there are 2 intervals. Pick a 'nice' step near (range / 2)
        step = _nice_step((raw_max - raw_min) / 2)
        ymin = np.floor(raw_min / step) * step
        ymax = np.ceil(raw_max / step) * step

        # Ensure the span fits exactly 2*step so that the midpoint is also nice.
        # Expand outward in step increments as needed.
        span_steps = int(np.ceil((ymax - ymin) / step))
        if span_steps <= 0:
            span_steps = 1
        if span_steps % 2 == 1:
            span_steps += 1  # make it even

        ymax = ymin + span_steps * step

        # If we expanded too much, try shifting up one step (without losing coverage)
        # to stay closer to data while keeping the 2*step structure.
        while (ymin + step) <= raw_min and (ymax + step) <= hi_clip:
            ymin += step
            ymax += step

    ymin = max(ymin, lo_clip)
    ymax = min(ymax, hi_clip)

    # Recompute ticks (exactly 3)
    ticks = np.array([ymin, ymin + (ymax - ymin) / 2, ymax])
    return ymin, ymax, ticks


def _scalar(v):
    """Stochastic modules (Decision-Tree, Exp-Tree, Shallow-Tree, IDS) store per-key
    values as {'mean', 'std', 'values'} (via `aggregate_trials`); deterministic
    modules (PEC/dscluster, ExKMC, WRA, CBA, CN2) store a bare float/int. Normalize
    to a scalar (the mean); treat None (JSON-encoded NaN) as missing too.
    """
    if isinstance(v, dict):
        return v.get('mean', np.nan)
    return v if v is not None else np.nan

### Generic Data Loader

In [ ]:
from experiments.cli_utils import conf_tag


def load_experiment(dataset, stage, confidence, ref='resub', prefix='exp', root='../data/experiments'):
    """Load one confidence-tagged experiment JSON.

    stage: 'alphas' | 'max_rules' | 'lambda'.
    prefix: 'exp' for the sweep results file, 'selected_alphas' for the
        externally elbow-selected alpha file (alphas stage only).

    Returns None (and prints a one-line notice) if the file doesn't exist, so
    callers can skip a dataset/threshold instead of crashing -- only a subset
    of datasets have been run through the confidence-tagged pipeline so far.
    """
    path = f"{root}/{dataset}/{stage}/{prefix}_{ref}_conf_{conf_tag(confidence)}.json"
    if not os.path.exists(path):
        print(f"[{stage}] skip {dataset} @ confidence={confidence} ({conf_tag(confidence)}): {path} not found")
        return None
    with open(path) as f:
        return json.load(f)


def pec_display_name(module):
    """'dscluster; {objective}; ensemble[; algorithm]' -> 'dscluster; ensemble[; algorithm]'.

    Strips the objective-name segment out of a PEC module's full name to get
    its display/style-dict lookup key. Used everywhere a PEC module needs to
    be looked up in color_dict/title_dict/marker_style_dict/linestyle_dict,
    replacing the several equivalent-but-inconsistent inline string-splits
    used ad hoc across sections of the original notebook.
    """
    parts = [p.strip() for p in module.split(';')]
    return '; '.join([parts[0]] + parts[2:])

## Global Configuration

In [ ]:
# --- Confidence used by Alphas / Max Rules / Bar / Bicriteria / Weighted-Avg / Uncertainty ---
SELECTED_CONFIDENCE = 0.5

# Datasets to attempt to load for all SELECTED_CONFIDENCE-driven sections below.
# Loading is defensive (see load_experiment) -- a dataset missing its
# confidence-tagged pipeline output is skipped with a printed notice rather
# than raising.
DATASETS = ["aniso", "anuran", "climate", "protein", "yeast", "mnist", "fashion"]

REF = "resub"
objective_names = ['coverage-cost', 'coverage-mistake', 'coverage-pairwise-distance']

# --- Lambdas section: bulk-load config (the section itself additionally
# selects one dataset/objective to actually plot -- see below) ---
CONFIDENCE_THRESHOLDS = [0.25, 0.5, 0.75]
LAMBDA_DATASETS = DATASETS

## Alphas

Loads and visualizes the `alphas.py` sweep at `SELECTED_CONFIDENCE`, plus the alpha value already
selected externally by `select_alphas.py` (elbow method). Alpha *selection* no longer happens in
this notebook -- this section only loads and displays the result.

In [ ]:
dataset_alpha_experiment_dict = {}
dataset_selected_alpha_dict = {}
for dataset in DATASETS:
    exp = load_experiment(dataset, 'alphas', SELECTED_CONFIDENCE, ref=REF, prefix='exp')
    sel = load_experiment(dataset, 'alphas', SELECTED_CONFIDENCE, ref=REF, prefix='selected_alphas')
    if exp is None or sel is None:
        continue
    dataset_alpha_experiment_dict[dataset] = exp
    dataset_selected_alpha_dict[dataset] = sel

dscluster_modules = (
    list(next(iter(dataset_alpha_experiment_dict.values()))['modules'].keys())
    if dataset_alpha_experiment_dict else []
)

In [ ]:
# Collect experiment data. The alpha-sweep curve (x, y, z) is computed exactly
# as in the alpha-selection logic; the "selected" alpha marker now comes from
# the externally selected_alphas file rather than being recomputed here.

alpha_objective_dict = {
    dataset: {objective: {} for objective in objective_names}
    for dataset in dataset_alpha_experiment_dict.keys()
}
for dataset, experiment_dict in dataset_alpha_experiment_dict.items():
    fixed_parameters = experiment_dict['fixed-parameters']
    selected_alphas = dataset_selected_alpha_dict[dataset]
    for objective in objective_names:
        selected_dscluster_module = [
            m for m in dscluster_modules if objective == m.split(';')[1].strip()
        ][0]  # there should only be one per objective!!
        reward = objective_cost_reward_dict[objective]['reward']
        cost = objective_cost_reward_dict[objective]['cost']

        alpha_vals = np.array(list(experiment_dict['modules'][selected_dscluster_module][cost].keys()), dtype=float)
        z = alpha_vals
        x = np.array([experiment_dict['modules'][selected_dscluster_module]['weighted-avg-length'][str(l)] for l in z])

        rl = np.array([experiment_dict['modules'][selected_dscluster_module]['sum-rule-length'][str(l)] for l in z])
        y1 = np.array([experiment_dict['modules'][selected_dscluster_module][reward][str(l)] for l in z])
        y2 = np.array([experiment_dict['modules'][selected_dscluster_module][cost][str(l)] for l in z]) + z * rl
        lambda_vals = np.array([experiment_dict['modules'][selected_dscluster_module]['lambda'][str(l)] for l in z])
        y = y1 - lambda_vals * y2

        selected_alpha = selected_alphas[selected_dscluster_module]
        selected_alpha_idx = int(np.argmin(np.abs(z - selected_alpha)))

        alpha_objective_dict[dataset][objective] = {
            'alpha_vals': z,
            'x': x,
            'y': y / fixed_parameters['n'],
            'selected_alpha_idx': selected_alpha_idx,
            'selected_alpha': selected_alpha,
        }

In [ ]:
# Plot results:

fig, axs = plt.subplots(
    len(objective_names), len(dataset_alpha_experiment_dict),
    figsize=(34, 14), squeeze=False,
)

for i, (dataset, objective_result_dict) in enumerate(alpha_objective_dict.items()):
    for j, (objective, module_result_dict) in enumerate(objective_result_dict.items()):
        ax = axs[j][i]

        # Gridlines:
        ax.grid(which='major', linestyle='-', linewidth=0.8, alpha=0.5)
        ax.grid(which='minor', linestyle=':', linewidth=0.8, alpha=0.5)
        ax.minorticks_on()

        # Plot the main line:
        x = module_result_dict['x']
        y = module_result_dict['y']
        z = module_result_dict['alpha_vals']
        selected_idx = module_result_dict['selected_alpha_idx']
        selected_alpha = module_result_dict['selected_alpha']

        points = np.array([x, y]).T.reshape(-1, 1, 2)
        segments = np.concatenate([points[:-1], points[1:]], axis=1)
        norm = plt.Normalize(z.min(), z.max())  # Normalize color map
        lc = LineCollection(segments, cmap='coolwarm', norm=norm)
        lc.set_array(z)
        lc.set_linewidth(6)
        line = ax.add_collection(lc)

        # Plot selected point (selected externally by select_alphas.py):
        ax.scatter(
            x[selected_idx],
            y[selected_idx],
            color='black',
            marker='o',
            s=200,
            zorder=5,
            label=rf'Selected $\alpha$ = {selected_alpha:.2g}'
        )

        # Set x and y ticks to make the plot look nice
        x_min, x_max, x_std = np.min(x), np.max(x), np.std(x)
        y_min, y_max, y_std = np.min(y), np.max(y), np.std(y)

        x_lo, x_hi, xticks = nice_lim_for_3_ticks(x_min - 0.001 * x_std, x_max + 0.001 * x_std, clip=(1.0, 6.0))
        y_lo, y_hi, yticks = nice_lim_for_3_ticks(y_min - 0.001 * y_std, y_max + 0.001 * y_std)

        ax.set_xlim(x_lo, x_hi)
        ax.set_xticks(xticks)
        ax.xaxis.set_major_formatter(FormatStrFormatter('%.2f'))

        ax.set_ylim(y_lo, y_hi)
        ax.set_yticks(yticks)
        ax.yaxis.set_major_formatter(FormatStrFormatter('%.2f'))

        # Increase tick-label padding away from the axes to reduce overlap at origin
        ax.tick_params(axis='x', which='major', pad=10)
        ax.tick_params(axis='y', which='major', pad=10)

        # Individual colorbar per subplot
        cbar = fig.colorbar(line, ax=ax, fraction=0.046, pad=0.02)
        cbar.locator = MaxNLocator(nbins=4)
        cbar.formatter.set_powerlimits((0, 0))  # always scientific (10^k)
        cbar.update_ticks()
        cbar.ax.yaxis.get_offset_text().set(size=18)

        # Y-label with objective
        if i == 0:
            ax.set_ylabel(rf"$\bar{{f}}_{function_name_dict[j]}$", rotation=0, labelpad=40, fontsize=36)

        # Title with dataset name
        if j == 0:
            ax.set_title(rf"${dataset.capitalize()}$")

fig.supxlabel(r"$\textup{weighted-avg-length}$", y=0.05, x=0.525)
plt.tight_layout()

#plt.savefig(
#    f"../figures/experiments/v2_alphas_conf{conf_tag(SELECTED_CONFIDENCE)}.pdf",
#    bbox_inches='tight',
#    dpi=300
#)

## Max Rules Experiment

Loads the `max_rules.py` sweep at `SELECTED_CONFIDENCE` for each dataset in `DATASETS`.

In [ ]:
# Load experiment data. Loading is defensive -- datasets whose max_rules.py
# output hasn't been generated yet at this confidence are skipped with a
# warning rather than raising.

dscluster_modules = set()
dataset_experiment_dict = {}
for dataset in DATASETS:
    experiment_dict = load_experiment(dataset, 'max_rules', SELECTED_CONFIDENCE, ref=REF, prefix='exp')
    if experiment_dict is None:
        continue
    dataset_experiment_dict[dataset] = experiment_dict
    dscluster_modules.update(m for m in experiment_dict['modules'] if 'dscluster' in m)
    missing = [m for m in comparison_modules if m not in experiment_dict['modules']]
    if missing:
        print(f"[max_rules] warning: {dataset} is missing modules {missing}")

dscluster_modules = list(dscluster_modules)
baseline_module = "KMeans"

In [ ]:
lambda_val_dict = {}
for dataset, experiment_dict in dataset_experiment_dict.items():
    if dataset not in lambda_val_dict:
        lambda_val_dict[dataset] = {}
    for module in experiment_dict['modules'].keys():
        if 'dscluster' not in module:
            continue
        lambda_val_dict[dataset][module] = list(experiment_dict['modules'][module]['lambda'].values())[0]

In [ ]:
# Lambda (lambda*) values for each dataset and objective:
pd.DataFrame(lambda_val_dict)

In [ ]:
alpha_val_dict = {}
for dataset, experiment_dict in dataset_experiment_dict.items():
    alpha_val_dict[dataset] = experiment_dict['fixed-parameters']['alpha']

In [ ]:
# Alpha values for each dataset and objective:
pd.DataFrame(alpha_val_dict)

### Bar Plots

In [ ]:
# Collect experiment data for bar plots:

bar_dict = {
    dataset: {objective: {} for objective in objective_names} for dataset in dataset_experiment_dict.keys()
}
for dataset, experiment_dict in dataset_experiment_dict.items():
    fixed_parameters = experiment_dict['fixed-parameters']
    for objective in objective_names:
        selected_dscluster_module = [
            m for m in dscluster_modules if objective == m.split(';')[1].strip()
        ][0]  # there should only be one per objective!!
        reward = objective_cost_reward_dict[objective]['reward']
        cost = objective_cost_reward_dict[objective]['cost']
        alpha = fixed_parameters['alpha'][selected_dscluster_module]
        lambd = max(list(experiment_dict['modules'][selected_dscluster_module]['lambda'].values()))

        x = np.array(list(experiment_dict['modules'][selected_dscluster_module][reward].keys()))
        idxs = np.where(x.astype(int) <= min(np.max(x.astype(int)), np.min(x.astype(int) + 6)))[0][::2]

        # Compute objective values:
        for cmod in comparison_modules:
            if cmod not in experiment_dict['modules']:
                continue
            obj1 = np.array([_scalar(v) for v in experiment_dict['modules'][cmod][reward].values()])
            obj2 = np.array([_scalar(v) for v in experiment_dict['modules'][cmod][cost].values()])
            obj3 = np.array([_scalar(v) for v in experiment_dict['modules'][cmod]['sum-rule-length'].values()])
            obj_values = obj1 - lambd * (obj2 + alpha * obj3)
            bar_dict[dataset][objective][cmod] = (x[idxs], obj_values[idxs] / fixed_parameters['n'])

        # DSCluster Module
        obj1 = np.array([_scalar(v) for v in experiment_dict['modules'][selected_dscluster_module][reward].values()])
        obj2 = np.array([_scalar(v) for v in experiment_dict['modules'][selected_dscluster_module][cost].values()])
        obj3 = np.array([_scalar(v) for v in experiment_dict['modules'][selected_dscluster_module]['sum-rule-length'].values()])
        obj_values = obj1 - lambd * (obj2 + alpha * obj3)
        bar_dict[dataset][objective][selected_dscluster_module] = (x[idxs], obj_values[idxs] / fixed_parameters['n'])

In [ ]:
# Plot results

fig, ax = plt.subplots(len(objective_names), len(dataset_experiment_dict), figsize=(34, 12), squeeze=False)

bar_module_order = list(module_order)  # local copy; last entry gets re-pointed per objective below

for i, (dataset, objective_result_dict) in enumerate(bar_dict.items()):
    for j, (objective, module_result_dict) in enumerate(objective_result_dict.items()):
        axj = ax[j][i]
        bar_module_order[-1] = f'dscluster; {objective}; ensemble'

        obj_min, obj_max = np.inf, -np.inf
        for k, module in enumerate(bar_module_order):
            if module not in module_result_dict:
                continue
            x, obj_values = module_result_dict[module]
            mod_name = pec_display_name(module) if 'dscluster' in module else module
            hatch = hatch_dict.get(mod_name, '')

            # Plot bar for a single module:
            width = 0.225
            axj.bar(
                x.astype(int) + k * width,
                obj_values,
                width=width,
                label=mod_name,
                color=color_dict.get(mod_name, 'grey'),
                hatch=hatch,
                edgecolor='black',
            )

            obj_min = min(obj_min, np.min(obj_values))
            obj_max = max(obj_max, np.max(obj_values))

        # Set y limits and ticks
        obj_rng = obj_max - obj_min
        pad = 0.01 * obj_rng
        y_lo, y_hi, yticks = nice_lim_for_3_ticks(obj_min - pad, obj_max + pad, clip=(0, 1.0))
        y_lo = 0.5
        axj.set_ylim(y_lo, y_hi)
        axj.set_yticks(yticks)
        axj.yaxis.set_major_formatter(FormatStrFormatter('%.2f'))

        # Set x ticks and labels
        axj.set_xticks(x.astype(int) + (len(module_result_dict) - 1) * 0.2 / 2)
        if j != len(objective_names) - 1:
            axj.set_xticklabels([])
        else:
            axj.set_xticklabels([str(int(val)) for val in x.astype(int)])

        # Y-label with objective
        if i == 0:
            axj.set_ylabel(rf"$\bar{{f}}_{function_name_dict[j]}$", rotation=0, labelpad=40, fontsize=36)

        # Title with dataset name
        if j == 0:
            axj.set_title(rf"${dataset.capitalize()}$")

        # Gridlines:
        axj.grid(which='major', linestyle='-', linewidth=0.8, alpha=0.5)
        axj.axhline(0.0, color="black", linewidth=2.0, alpha=0.7)

fig.supxlabel(r"Number of Rules, $\ell$", y=0.05, x=0.525)
plt.tight_layout()

#plt.savefig(
#    f"../figures/experiments/v2_objectives_conf{conf_tag(SELECTED_CONFIDENCE)}.pdf",
#    bbox_inches='tight',
#    dpi=300
#)

In [ ]:
# Create a separate legend for the bar plot with hatch patterns

fig, ax = plt.subplots(figsize=(10, 2))

legend_elements = []
for mod in module_order:
    # module_order already holds display names (e.g. 'dscluster; ensemble'),
    # not raw JSON module keys -- no pec_display_name translation needed here.
    mod_name = mod
    legend_elements.append(
        mpatches.Patch(
            facecolor=color_dict[mod_name],
            edgecolor='black',
            linewidth=1.5,
            hatch=hatch_dict.get(mod_name, ''),
            label=title_dict.get(mod_name, mod_name),
        )
    )

ax.legend(handles=legend_elements, ncol=len(module_order), loc='center', frameon=False,
          handlelength=2, handleheight=2)
ax.axis('off')

plt.show()

### Bicriteria Plots

In [ ]:
# Collect experiment data for scatter plots:
#
# Rather than folding the cost and rule-length objectives into a single
# lambda-weighted axis, we keep all three objectives (obj1 = coverage/reward,
# obj2 = cost, obj3 = summed rule length) separate and min-max scale each to
# [0, 1]. Normalization constants are computed jointly across every module for
# a given (dataset, objective) pair, so the scaled values stay comparable
# across modules within a subplot.

scatter_dict = {
    dataset: {objective: {} for objective in objective_names} for dataset in dataset_experiment_dict.keys()
}
for dataset, experiment_dict in dataset_experiment_dict.items():
    for objective in objective_names:
        selected_dscluster_module = [
            m for m in dscluster_modules if objective == m.split(';')[1].strip()
        ][0]  # there should only be one per objective!!
        reward = objective_cost_reward_dict[objective]['reward']
        cost = objective_cost_reward_dict[objective]['cost']

        keys = np.array(list(experiment_dict['modules'][selected_dscluster_module][reward].keys()))
        idxs = np.where(keys.astype(int) <= min(np.max(keys.astype(int)), np.min(keys.astype(int) + 6)))[0][::2]

        # Gather raw (unnormalized) objective values per module first, so that
        # normalization constants can be computed jointly across all modules.
        raw_values = {}
        modules_to_process = [cmod for cmod in comparison_modules if cmod in experiment_dict['modules']]
        modules_to_process.append(selected_dscluster_module)
        for mod in modules_to_process:
            obj1 = np.array([_scalar(v) for v in experiment_dict['modules'][mod][reward].values()])
            obj2 = np.array([_scalar(v) for v in experiment_dict['modules'][mod][cost].values()])
            obj3 = np.array([_scalar(v) for v in experiment_dict['modules'][mod]['sum-rule-length'].values()])
            raw_values[mod] = (obj1, obj2, obj3)

        # Min-max normalization constants, shared across modules so scaled
        # values remain directly comparable within this (dataset, objective).
        obj1_lo = min(v[0].min() for v in raw_values.values())
        obj1_hi = max(v[0].max() for v in raw_values.values())
        obj2_lo = min(v[1].min() for v in raw_values.values())
        obj2_hi = max(v[1].max() for v in raw_values.values())
        obj3_lo = min(v[2].min() for v in raw_values.values())
        obj3_hi = max(v[2].max() for v in raw_values.values())

        def _minmax(v, lo, hi):
            rng = hi - lo
            return (v - lo) / rng if rng > 0 else np.zeros_like(v)

        for mod, (obj1, obj2, obj3) in raw_values.items():
            x = _minmax(obj1, obj1_lo, obj1_hi)
            y = _minmax(obj2, obj2_lo, obj2_hi)
            z = _minmax(obj3, obj3_lo, obj3_hi)
            scatter_dict[dataset][objective][mod] = (x[idxs], y[idxs], z[idxs])

#### Option 1: Marker-Size Encoding

`x` = obj1 (coverage/reward), `y` = obj2 (cost), both scaled to `[0, 1]`. Marker size encodes obj3
(summed rule length), **inverted** relative to the natural mapping: the smallest obj3 (fewest/shortest
rules -- the more interpretable outcome) gets the *largest* marker, so the more desirable solutions
visually pop out of the plot.

In [ ]:
# Plot results (Option 1): marker size encodes (1 - obj3), inverted so the
# smallest obj3 (fewest/shortest rules) gets the largest marker.

# Marker area range (in points^2) that (1 - obj3) in [0, 1] is mapped onto. A
# nonzero floor keeps the largest-obj3 points visible instead of vanishing to
# a point.
SIZE_MIN, SIZE_MAX = 80, 500

def _size_from_z(z):
    # z is already min-max scaled so z=0 == smallest raw obj3 (sum-rule-length).
    # INVERTED vs. the natural mapping: smallest obj3 (fewest/shortest rules)
    # gets the LARGEST marker, since fewer/shorter rules is the more
    # interpretable/desirable outcome and should visually stand out.
    return SIZE_MIN + (SIZE_MAX - SIZE_MIN) * (1 - z)

fig, axs = plt.subplots(len(objective_names), len(dataset_experiment_dict), figsize=(34, 14), squeeze=False)

for i, (dataset, objective_result_dict) in enumerate(scatter_dict.items()):
    for j, (objective, module_result_dict) in enumerate(objective_result_dict.items()):
        ax = axs[j][i]

        # Collect x/y values across modules to set shared limits per subplot
        x_vals_all = np.array([x for _, (x, _, _) in module_result_dict.items()])
        y_vals_all = np.array([y for _, (_, y, _) in module_result_dict.items()])
        x_min, x_max = np.min(x_vals_all), np.max(x_vals_all)
        y_min, y_max = np.min(y_vals_all), np.max(y_vals_all)

        # Scatter points for each module
        for module, (x, y, z) in module_result_dict.items():
            mod_name = pec_display_name(module) if 'dscluster' in module else module

            ax.scatter(
                x, y,
                s=_size_from_z(z),
                label=mod_name,
                color=color_dict.get(mod_name, 'grey'),
                marker=marker_style_dict.get(mod_name, 'o'),
                edgecolor='black',
                alpha=0.9,
            )

        # X and Y ticks
        x_rng, y_rng = float(x_max - x_min), float(y_max - y_min)
        x_pad, y_pad = 0.05 * x_rng, 0.05 * y_rng
        # x, y are min-max scaled to [0, 1] by construction; clip ticks/limits to that range
        x_lo, x_hi, xticks = nice_lim_for_3_ticks(x_min - x_pad, x_max + x_pad, clip=(0.0, 1.0))
        y_lo, y_hi, yticks = nice_lim_for_3_ticks(y_min - y_pad, y_max + y_pad, clip=(0.0, 1.0))

        ax.set_xlim(x_lo, x_hi)
        ax.set_xticks(xticks)
        ax.xaxis.set_major_formatter(FormatStrFormatter('%.2f'))

        ax.set_ylim(y_lo, y_hi)
        ax.set_yticks(yticks)
        ax.yaxis.set_major_formatter(FormatStrFormatter('%.2f'))

        # Increase tick-label padding away from the axes to reduce overlap at origin
        ax.tick_params(axis='x', which='major', pad=10)
        ax.tick_params(axis='y', which='major', pad=10)

        # Title with dataset name
        if j == 0:
            ax.set_title(rf"${dataset.capitalize()}$")

        # Y-label with objective
        if i == 0:
            ax.set_ylabel(rf"$\bar{{h}}_{function_name_dict[j]}$", rotation=0, labelpad=40, fontsize=36)

        # Gridlines:
        ax.grid(which='major', linestyle='-', linewidth=0.8, alpha=0.5)
        ax.grid(which='minor', linestyle=':', linewidth=0.8, alpha=0.5)
        ax.minorticks_on()

fig.supxlabel(r"$\bar{g}$", y=0.05, x=0.525)
plt.tight_layout()

#plt.savefig(
#    f"../figures/experiments/v2_bicriteria_marker_size_conf{conf_tag(SELECTED_CONFIDENCE)}.pdf",
#    bbox_inches='tight',
#    dpi=300
#)

In [ ]:
# Marker-size legend for (1 - obj3), shared by all Option-1 subplots.
fig, ax = plt.subplots(figsize=(6, 1.6))

z_ref = [0.0, 0.5, 1.0]  # scaled obj3 (sum-rule-length) reference values
for k, z in enumerate(z_ref):
    ax.scatter(
        [k], [0],
        s=_size_from_z(z),
        color='white',
        edgecolor='black',
        linewidth=1.5,
    )
    ax.annotate(f"{z:.1f}", (k, 0), xytext=(0, -28), textcoords='offset points',
                ha='center', va='top', fontsize=20)

ax.set_xlim(-0.5, len(z_ref) - 0.5)
ax.set_ylim(-1, 1)
# Inverted vs. the usual "size grows with value" convention, so the direction
# is spelled out explicitly rather than left implicit in "marker size ∝ obj3".
ax.set_title(r"marker size $\propto$ (1 $-$ obj3); larger marker = fewer/shorter rules", fontsize=18, pad=10)
ax.axis('off')

plt.show()

#### Option 2: 3D Scatter

All three axes (`x` = obj1, `y` = obj2, `z` = obj3) are shown directly, each scaled to `[0, 1]`. To
make each point's position easier to read off a static image, every point is also drawn as a faded
shadow on the floor plane (`z=0`) with a thin drop-line connecting it to its real height, and the
wall panes carry stronger shading/gridlines as a spatial reference frame. Fixed viewing angle,
matching color/marker encoding as Option 1, to stay legible in print.

In [ ]:
# Plot results (Option 2): 3D scatter over (obj1, obj2, obj3).

from mpl_toolkits.mplot3d import Axes3D  # noqa: F401 (registers the '3d' projection)

fig, axs = plt.subplots(
    len(objective_names), len(dataset_experiment_dict),
    figsize=(34, 14), subplot_kw={'projection': '3d'}, squeeze=False,
)

for i, (dataset, objective_result_dict) in enumerate(scatter_dict.items()):
    for j, (objective, module_result_dict) in enumerate(objective_result_dict.items()):
        ax = axs[j][i]

        for module, (x, y, z) in module_result_dict.items():
            mod_name = pec_display_name(module) if 'dscluster' in module else module
            color = color_dict.get(mod_name, 'grey')
            marker = marker_style_dict.get(mod_name, 'o')

            # Floor-plane (z=0) shadow projection: same (x, y), desaturated,
            # drawn first so the real points render visually on top of it.
            ax.scatter(
                x, y, np.zeros_like(z),
                color=color, marker=marker, s=70, alpha=0.25,
                edgecolor='none', depthshade=False,
            )

            # Thin drop-lines connecting each real point to its floor shadow,
            # so height above the floor is legible without rotating the plot.
            for xi, yi, zi in zip(x, y, z):
                ax.plot([xi, xi], [yi, yi], [0, zi], color=color, linewidth=0.6, alpha=0.35, zorder=1)

            # Real point, drawn last/on top.
            ax.scatter(
                x, y, z,
                label=mod_name, color=color, marker=marker,
                s=120, edgecolor='black', alpha=0.9, depthshade=False, zorder=5,
            )

        ax.set_xlim(0, 1)
        ax.set_ylim(0, 1)
        ax.set_zlim(0, 1)
        ax.set_xticks([0, 0.5, 1])
        ax.set_yticks([0, 0.5, 1])
        ax.set_zticks([0, 0.5, 1])
        ax.tick_params(axis='both', which='major', labelsize=16, pad=0)

        ax.view_init(elev=22, azim=-60)

        # Stronger wall-pane shading + gridlines as a spatial reference frame,
        # to help judge where a point sits along each axis.
        ax.xaxis.pane.set_alpha(0.15); ax.xaxis.pane.set_facecolor((0.9, 0.9, 0.9))
        ax.yaxis.pane.set_alpha(0.15); ax.yaxis.pane.set_facecolor((0.9, 0.9, 0.9))
        ax.zaxis.pane.set_alpha(0.15); ax.zaxis.pane.set_facecolor((0.9, 0.9, 0.9))
        ax.grid(True, linestyle='-', linewidth=0.7, alpha=0.6)

        # Title with dataset name
        if j == 0:
            ax.set_title(rf"${dataset.capitalize()}$", pad=20)

        # Row label with objective
        if i == 0:
            ax.set_ylabel(rf"$\bar{{h}}_{function_name_dict[j]}$", labelpad=20, fontsize=22)

        ax.set_xlabel(r"$\bar{g}$", labelpad=14, fontsize=22)
        ax.set_zlabel(r"$\bar{p}$", labelpad=10, fontsize=22)

plt.tight_layout()

#plt.savefig(
#    f"../figures/experiments/v2_bicriteria_3d_conf{conf_tag(SELECTED_CONFIDENCE)}.pdf",
#    bbox_inches='tight',
#    dpi=300
#)

In [ ]:
# Create a separate legend for the scatter plots (shared by Option 1 and 2)
fig, ax = plt.subplots(figsize=(10, 2))

legend_elements = [
    mlines.Line2D(
        [], [],
        color=color_dict[m],
        marker=marker_style_dict.get(m, 'o'),
        markersize=30,
        markeredgecolor='k',
        markeredgewidth=1.5,
        linestyle='None',
        label=title_dict.get(m, m),
    )
    for m in module_order
]

ax.legend(handles=legend_elements, ncol=len(module_order), loc='center', frameon=False)
ax.axis('off')

plt.show()

### Weighted Average Rule Length

In [ ]:
# Collect the weighted average lengths for each dataset and objective.
weighted_avg_dict = {
    (objective, dataset): {}
    for dataset in dataset_experiment_dict.keys()
    for objective in objective_names
}

for dataset, experiment_dict in dataset_experiment_dict.items():
    for objective in objective_names:

        for cmod in comparison_modules:
            if cmod not in experiment_dict['modules']:
                continue
            cmod_weighted_avg_lengths = experiment_dict['modules'][cmod]['weighted-avg-length']
            cmod_weighted_avg_length = _scalar(cmod_weighted_avg_lengths[
                str(min(int(l) for l in cmod_weighted_avg_lengths.keys()))
            ])
            weighted_avg_dict[(objective, dataset)][cmod] = cmod_weighted_avg_length

        selected_dscluster_module = [
            m for m in dscluster_modules if objective == m.split(';')[1].strip()
        ][0]
        dmod_name = pec_display_name(selected_dscluster_module)
        dmod_weighted_avg_lengths = experiment_dict['modules'][selected_dscluster_module]['weighted-avg-length']
        dmod_weighted_avg_length = _scalar(dmod_weighted_avg_lengths[
            str(min(int(l) for l in dmod_weighted_avg_lengths.keys()))
        ])
        weighted_avg_dict[(objective, dataset)][dmod_name] = dmod_weighted_avg_length

In [ ]:
# Weighted rule length table:
pd.DataFrame(weighted_avg_dict)

### Uncertainty

In [ ]:
# Collect the distributions of covered weights for each dataset and objective.

distribution_dict = {
    dataset: {objective: None for objective in objective_names} for dataset in dataset_experiment_dict.keys()
}
for dataset, experiment_dict in dataset_experiment_dict.items():
    fixed_parameters = experiment_dict['fixed-parameters']

    for objective in objective_names:

        weights = np.array(fixed_parameters['weights'])

        selected_dscluster_module = [
            m for m in dscluster_modules if objective == m.split(';')[1].strip()
        ][0]  # there should only be one per objective!!
        mod_covered_sets = experiment_dict['modules'][selected_dscluster_module]['cluster-coverage-set']
        mod_covered_set = mod_covered_sets[
            str(min(int(l) for l in mod_covered_sets.keys()))
        ]
        mod_covered_weights = weights[mod_covered_set]

        # Not a display name -- builds the JSON lookup key for the *weighted*
        # sibling module, so this stays a positional split rather than being
        # routed through pec_display_name.
        selected_dscluster_weighted_module = (
            selected_dscluster_module.split(';')[0] + '; ' + objective
            + '-weighted;' + selected_dscluster_module.split(';')[2]
        )
        weighted_mod_covered_sets = experiment_dict['modules'][selected_dscluster_weighted_module]['cluster-coverage-set']
        weighted_mod_covered_set = weighted_mod_covered_sets[
            str(min(int(l) for l in weighted_mod_covered_sets.keys()))
        ]
        weighted_mod_covered_weights = weights[weighted_mod_covered_set]

        samples1 = np.log(weights[mod_covered_set]) / -5
        samples2 = np.log(weights[weighted_mod_covered_set]) / -5
        distribution_dict[dataset][objective] = (samples1, samples2)

In [ ]:
# Plot results:

fig, axs = plt.subplots(len(objective_names), len(dataset_experiment_dict), figsize=(34, 12), squeeze=False)

for i, (dataset, objective_result_dict) in enumerate(distribution_dict.items()):
    for j, (objective, (samples1, samples2)) in enumerate(objective_result_dict.items()):
        ax = axs[j][i]

        edges = np.linspace(0.0, 1.0, 31)  # 30 equal-width bins in [0, 1]
        h1, _ = np.histogram(samples1, bins=edges)
        h2, _ = np.histogram(samples2, bins=edges)

        # Convert counts to probabilities (mass per bin)
        p1 = h1 / max(h1.sum(), 1)
        p2 = h2 / max(h2.sum(), 1)
        diff = p2 - p1

        centers = 0.5 * (edges[:-1] + edges[1:])
        width = edges[1:] - edges[:-1]
        ax.bar(
            centers,
            diff,
            width=width,
            align="center",
            alpha=0.6,
            edgecolor="black",
            linewidth=2.0,
        )

        # Y-axis limits and ticks: choose ymin/ymax so 3 evenly spaced ticks are also 'nice'
        obj_min, obj_max = np.min(diff), np.max(diff)
        obj_rng = obj_max - obj_min
        pad = 0.01 * obj_rng

        ymin, ymax, yticks = nice_lim_for_3_ticks(obj_min - pad, obj_max + pad, clip=(-1.0, 1.0))
        ax.set_ylim(ymin, ymax)
        ax.set_yticks(yticks)
        ax.yaxis.set_major_formatter(FormatStrFormatter('%.3f'))

        # X-axis ticks and labels
        ax.set_xlim(0.0, 1.0)
        if j != len(objective_names) - 1:
            ax.set_xticklabels([])

        # Increase tick-label padding away from the axes to reduce overlap at origin
        ax.tick_params(axis='x', which='major', pad=10)
        ax.tick_params(axis='y', which='major', pad=10)

        # Title with dataset name
        if j == 0:
            ax.set_title(rf"${dataset.capitalize()}$")

        # Gridlines:
        ax.grid(which='major', linestyle='-', linewidth=0.8, alpha=0.5)
        ax.axhline(0.0, color="black", linewidth=2.0, alpha=0.7)

fig.supylabel(r"$\delta(\mathcal{W}, \mathcal{U})$", x=0.02, rotation=0)
fig.supxlabel(r"$\textup{distance-ratio}$", y=0.05, x=0.525)
plt.tight_layout()

#plt.savefig(
#    f"../figures/experiments/v2_uncertainty_conf{conf_tag(SELECTED_CONFIDENCE)}.pdf",
#    bbox_inches='tight',
#    dpi=300
#)

plt.show()

## Lambdas

Compares PEC's `lambda_val` sweep (`lambda.py`) against the fixed baselines across multiple
confidence thresholds side by side, for one selected dataset and PEC objective at a time.

The combined PEC-style objective score `y = (reward - lambda*(cost + alpha*sum_rule_length)) / n`
isn't stored directly in the sweep output, so it's reconstructed per model per lambda below (same
formula as the Bar Plots section). For the comparison models -- whose reward/cost don't depend on
lambda at all -- this simply traces a straight declining reference line, showing where PEC's
optimized curve crosses each fixed baseline as lambda increases.

The dashed vertical line marks $\lambda^*$, the minimum lambda for which the distorted-greedy
approximation guarantee holds; the dotted distorted-greedy curve is only evaluated to its right.

In [ ]:
# Load lambda.py results for every (confidence, dataset) pair. Defensive --
# most datasets have not been run through this pipeline stage yet.

lambda_data = {}  # lambda_data[confidence][dataset] = experiment_dict
for confidence in CONFIDENCE_THRESHOLDS:
    lambda_data[confidence] = {}
    for dataset in LAMBDA_DATASETS:
        exp = load_experiment(dataset, 'lambda', confidence, ref=REF, prefix='exp')
        if exp is None:
            continue
        lambda_data[confidence][dataset] = exp

In [ ]:
# Single dataset and PEC objective to actually plot below -- change and rerun
# the remaining cells in this section to inspect a different one.
LAMBDA_DATASET = "aniso"
LAMBDA_OBJECTIVE = "coverage-cost"

In [ ]:
# Collect: for each confidence threshold, reconstruct the combined objective
# score vs. lambda for PEC (lazy-greedy & distorted-greedy) and every
# comparison model, plus track global x/y ranges for shared axis limits.

reward_key = objective_cost_reward_dict[LAMBDA_OBJECTIVE]['reward']
cost_key = objective_cost_reward_dict[LAMBDA_OBJECTIVE]['cost']
pec_module = f'dscluster; {LAMBDA_OBJECTIVE}; ensemble'


def _combined_objective(mod, keys, alpha, n):
    """y = (reward - lambda*(cost + alpha*sum_rule_length)) / n at each lambda
    key in `keys`, read out of module dict `mod` via _scalar. Not stored
    directly in the sweep output -- reconstructed here from its components,
    the same formula used in the Bar Plots section above.
    """
    lam = np.array([float(k) for k in keys])
    reward = np.array([_scalar(mod[reward_key][k]) for k in keys])
    cost = np.array([_scalar(mod[cost_key][k]) for k in keys])
    rl = np.array([_scalar(mod['sum-rule-length'][k]) for k in keys])
    y = (reward - lam * (cost + alpha * rl)) / n
    order = np.argsort(lam)
    return lam[order], y[order]


lambda_lines = {}       # lambda_lines[confidence] = {module_name: (lam_arr, y_arr, linestyle)}
lambda_star_by_conf = {}
all_x, all_y = [], []

for confidence in CONFIDENCE_THRESHOLDS:
    exp = lambda_data[confidence].get(LAMBDA_DATASET)
    lambda_lines[confidence] = {}
    if exp is None:
        lambda_star_by_conf[confidence] = None
        continue

    fp = exp['fixed-parameters']
    alpha = fp['alpha'][pec_module]
    n = fp['n']
    lambda_star_by_conf[confidence] = fp['lambda_star'].get(pec_module)

    modules = exp['modules']

    # PEC lazy-greedy / distorted-greedy: reward/cost vary per lambda key,
    # since the rule set actually changes across the sweep.
    for algo, linestyle in [('lazy-greedy', 'solid'), ('distorted-greedy', 'dotted')]:
        mod_name = f'{pec_module}; {algo}'
        if mod_name not in modules:
            continue
        keys = list(modules[mod_name][reward_key].keys())  # this module's own grid
        lam, y = _combined_objective(modules[mod_name], keys, alpha, n)
        lambda_lines[confidence][mod_name] = (lam, y, linestyle)
        all_x.extend(lam); all_y.extend(y)

    # Comparison models: reward/cost are broadcast-constant across every key
    # in their dict (they don't depend on PEC's lambda at all), so we reuse
    # lazy-greedy's own key strings to index into each one -- verified those
    # keys are an exact-string subset of every comparison model's dict.
    lazy_mod_name = f'{pec_module}; lazy-greedy'
    if lazy_mod_name in modules:
        ref_keys = list(modules[lazy_mod_name][reward_key].keys())
        for cmod in comparison_modules:
            if cmod not in modules:
                continue
            lam, y = _combined_objective(modules[cmod], ref_keys, alpha, n)
            lambda_lines[confidence][cmod] = (lam, y, 'solid')
            all_x.extend(lam); all_y.extend(y)

x_lo, x_hi = (min(all_x), max(all_x)) if all_x else (0.0, 1.0)
y_lo, y_hi = (min(all_y), max(all_y)) if all_y else (0.0, 1.0)

In [ ]:
# Plot: one row, one subplot per confidence threshold (numerical order),
# shared x/y limits, lambda* as a dashed vertical reference line.

sorted_thresholds = sorted(CONFIDENCE_THRESHOLDS)

fig, axs = plt.subplots(1, len(sorted_thresholds), figsize=(8 * len(sorted_thresholds), 7), squeeze=False)

for t, confidence in enumerate(sorted_thresholds):
    ax = axs[0][t]
    lines = lambda_lines[confidence]

    if not lines:
        ax.text(0.5, 0.5, f"{LAMBDA_DATASET} missing\n@ confidence={confidence}",
                 ha='center', va='center', transform=ax.transAxes)
        ax.set_xticks([]); ax.set_yticks([])
        continue

    for name, (lam, y, linestyle) in lines.items():
        display = pec_display_name(name) if 'dscluster' in name else name
        color = color_dict.get('dscluster; ensemble', 'grey') if 'dscluster' in name else color_dict.get(name, 'grey')
        label = title_dict.get(display, display)
        if 'distorted-greedy' in name:
            label += ' (distorted)'
        ax.plot(lam, y, color=color, linestyle=linestyle, linewidth=4, label=label)

    lam_star = lambda_star_by_conf[confidence]
    if lam_star is not None:
        ax.axvline(lam_star, color='black', linestyle='dashed', linewidth=2, alpha=0.7, label=r'$\lambda^*$')

    ax.set_xlim(x_lo, x_hi)
    ax.set_ylim(y_lo, y_hi)
    ax.grid(which='major', linestyle='-', linewidth=0.8, alpha=0.5)
    ax.set_title(rf"confidence = {confidence}")
    ax.set_xlabel(r"$\lambda$")
    if t == 0:
        ax.set_ylabel(r"$\bar{f}$", rotation=0, labelpad=30)

fig.suptitle(
    rf"{LAMBDA_DATASET.capitalize()} -- {objective_name_dict.get(LAMBDA_OBJECTIVE, LAMBDA_OBJECTIVE)}", y=1.05
)
plt.tight_layout()

#plt.savefig(
#    f"../figures/experiments/v2_lambdas_{LAMBDA_DATASET}_{LAMBDA_OBJECTIVE}.pdf",
#    bbox_inches='tight',
#    dpi=300
#)

In [ ]:
# Create a separate legend for the Lambdas plots
fig, ax = plt.subplots(figsize=(12, 2))

legend_elements = [
    mlines.Line2D([], [], color=color_dict['dscluster; ensemble'], linestyle='solid', linewidth=4,
                  label=title_dict['dscluster; ensemble'] + ' (lazy-greedy)'),
    mlines.Line2D([], [], color=color_dict['dscluster; ensemble'], linestyle='dotted', linewidth=4,
                  label=title_dict['dscluster; ensemble'] + ' (distorted-greedy)'),
] + [
    mlines.Line2D([], [], color=color_dict.get(m, 'grey'), linestyle='solid', linewidth=4,
                  label=title_dict.get(m, m))
    for m in comparison_modules
] + [
    mlines.Line2D([], [], color='black', linestyle='dashed', linewidth=2, label=r'$\lambda^*$'),
]

ax.legend(handles=legend_elements, ncol=4, loc='center', frameon=False)
ax.axis('off')

plt.show()